# T04 - 可转债研究

## 项目概述

可转债研究是一个综合性的债券分析项目，专注于可转债的数据采集、分析和策略研究。

### 功能描述
- 从同花顺iFinD采集可转债实时行情数据
- 构建多维度指数体系（Nature、Premium、Industry）
- 进行可转债投资策略研究

### 数据源
- 同花顺iFinD数据接口
- MySQL数据库（bond、yq、stock库）

### 输出结果
- 可转债指数数据（行业指数、债性分类指数、溢价分类指数）
- 正股指数数据
- 统计分析报告

---

## 1. 环境配置

### 1.1 导入依赖

In [ ]:
# 基础库
import os
import sys
import logging
import time
import json
from pathlib import Path
from typing import List, Dict, Optional, Literal
from datetime import datetime, timedelta
from decimal import Decimal

# 数据处理
import pandas as pd
import numpy as np
from tqdm import tqdm

# 数据库
import pymysql
from sqlalchemy import create_engine, text
import threading
import queue
import yaml

# 可视化
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# 同花顺iFinD（需要安装iFinDPy）
try:
    from iFinDPy import *
    THS_AVAILABLE = True
except ImportError:
    THS_AVAILABLE = False
    print("警告: iFinDPy未安装，同花顺数据采集功能不可用")

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('bond_research')

print("环境配置完成！")

### 1.2 配置参数

In [ ]:
# 从config模块导入配置
from config import (
    DB_CONFIG, 
    THS_CONFIG, 
    COLLECTOR_CONFIG,
    INDEX_CONFIG
)

# 显示关键配置（隐藏敏感信息）
print("数据库配置:")
print(f"  主机: {DB_CONFIG.get('host', 'N/A')}")
print(f"  端口: {DB_CONFIG.get('port', 'N/A')}")
print(f"  数据库: {DB_CONFIG.get('database', 'N/A')}")
print(f"\n同花顺配置: 用户名 = {THS_CONFIG.get('username', 'N/A')}")
print(f"\n指数配置:")
for index_type, configs in INDEX_CONFIG.items():
    print(f"  {index_type}: {list(configs.keys())}")

---

## 2. 数据获取

### 2.1 数据库连接池

In [ ]:
class DatabaseConnectionPool:
    """数据库连接池管理"""
    _instance = None
    _lock = threading.Lock()
    
    def __new__(cls):
        with cls._lock:
            if cls._instance is None:
                cls._instance = super(DatabaseConnectionPool, cls).__new__(cls)
            return cls._instance
    
    def __init__(self):
        if not hasattr(self, 'initialized'):
            self.config = DB_CONFIG
            self.max_connections = 10
            self.connections = queue.Queue(maxsize=self.max_connections)
            self.active_connections = 0
            self.initialized = True
            self._fill_pool()
    
    def _create_connection(self) -> pymysql.Connection:
        """创建新的数据库连接"""
        db_config = self.config.copy()
        retry_count = 3
        retry_delay = 1
        
        for attempt in range(retry_count):
            try:
                return pymysql.connect(**db_config)
            except Exception as e:
                if attempt == retry_count - 1:
                    raise
                time.sleep(retry_delay)
    
    def _fill_pool(self):
        """填充连接池"""
        min_connections = 2
        while self.active_connections < min_connections:
            try:
                conn = self._create_connection()
                self.connections.put(conn)
                self.active_connections += 1
            except Exception as e:
                logger.error(f"填充连接池失败: {str(e)}")
                break
    
    def get_connection(self) -> pymysql.Connection:
        """获取数据库连接"""
        try:
            conn = self.connections.get_nowait()
            try:
                conn.ping(reconnect=True)
                return conn
            except:
                self.active_connections -= 1
                conn = self._create_connection()
                self.active_connections += 1
                return conn
        except queue.Empty:
            if self.active_connections < self.max_connections:
                conn = self._create_connection()
                self.active_connections += 1
                return conn
            else:
                return self.connections.get()
    
    def return_connection(self, conn: pymysql.Connection):
        """归还数据库连接"""
        try:
            conn.ping(reconnect=False)
            self.connections.put(conn)
        except:
            try:
                conn.close()
            except:
                pass
            self.active_connections -= 1
            self._fill_pool()


class DatabaseConnection:
    """数据库连接上下文管理器"""
    def __init__(self):
        self.pool = DatabaseConnectionPool()
        self.conn = None
    
    def __enter__(self) -> pymysql.Connection:
        self.conn = self.pool.get_connection()
        return self.conn
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.conn:
            self.pool.return_connection(self.conn)


# 测试数据库连接
try:
    with DatabaseConnection() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT 1")
        result = cursor.fetchone()
        print(f"数据库连接测试: {'成功' if result else '失败'}")
except Exception as e:
    print(f"数据库连接失败: {e}")

### 2.2 同花顺数据采集器

In [ ]:
class THSBondDataCollector:
    """同花顺可转债数据采集器"""
    
    def __init__(self):
        self.logger = logging.getLogger('bond_collector')
        self.engine = self._create_db_engine()
        self.login_status = False
        
        # 创建数据存储目录
        self.raw_data_dir = os.path.join('data', 'raw')
        self.processed_data_dir = os.path.join('data', 'processed')
        os.makedirs(self.raw_data_dir, exist_ok=True)
        os.makedirs(self.processed_data_dir, exist_ok=True)
        
        # 初始化连接
        if THS_AVAILABLE:
            self.connect()
    
    def _create_db_engine(self):
        """创建数据库引擎"""
        conn_str = f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
        return create_engine(conn_str)
    
    def connect(self):
        """连接同花顺数据接口"""
        if not THS_AVAILABLE:
            self.logger.warning("同花顺iFinD不可用")
            return
            
        try:
            if self.login_status:
                THS_iFinDLogout()
                self.login_status = False
                
            retry_count = 0
            while not self.login_status and retry_count < 3:
                login_result = THS_iFinDLogin(THS_CONFIG['username'], THS_CONFIG['password'])
                if login_result == 0:
                    self.login_status = True
                    self.logger.info("同花顺数据接口连接成功")
                    break
                else:
                    retry_count += 1
                    self.logger.error(f"同花顺连接失败，错误码：{login_result}")
                    time.sleep(5)
                    
        except Exception as e:
            self.logger.error(f"连接同花顺数据接口时发生错误: {str(e)}")
    
    def get_bond_codes(self) -> List[str]:
        """获取需要采集的可转债代码"""
        try:
            query = """
                SELECT DISTINCT trade_code
                FROM bond.marketinfo_equity 
                WHERE ths_bond_balance_bond > 0
                AND dt >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
            """
            with self.engine.connect() as conn:
                df = pd.read_sql(query, conn)
                return df['trade_code'].tolist()
        except Exception as e:
            self.logger.error(f"获取可转债代码失败: {str(e)}")
            return []
    
    def collect_bond_data(self, bond_code: str, start_date: str, end_date: str) -> Optional[pd.DataFrame]:
        """采集单个可转债的数据"""
        if not THS_AVAILABLE or not self.login_status:
            self.logger.warning("同花顺未连接，无法采集数据")
            return None
            
        try:
            # 采集指标配置
            indicators_config = [
                {'name': 'ths_bond_close_cbond', 'param': '103'},  # 收盘价
                {'name': 'ths_new_bond_amt_cbond', 'param': ''},   # 成交额
                {'name': 'ths_pure_bond_premium_rate_cbond', 'param': ''},  # 纯债溢价率
                {'name': 'ths_pure_bond_ytm_cbond', 'param': ''},  # 纯债到期收益率
                {'name': 'ths_conversion_premium_rate_cbond', 'param': ''},  # 转股溢价率
                {'name': 'ths_conver_pe_cbond', 'param': ''},  # 转股市盈率
                {'name': 'ths_stock_pe_cbond', 'param': ''},   # 正股市盈率
                {'name': 'ths_stock_pb_cbond', 'param': ''},   # 正股市净率
                {'name': 'ths_conver_pb_cbond', 'param': ''}   # 转股市净率
            ]
            
            all_data = []
            for indicator in indicators_config:
                result = THS_DS(bond_code, indicator['name'], indicator['param'], '', start_date, end_date)
                if result.data is not None and not result.data.empty:
                    all_data.append(result.data)
                time.sleep(0.5)
            
            if not all_data:
                return None
            
            # 合并数据
            final_df = all_data[0]
            for df in all_data[1:]:
                final_df = pd.merge(final_df, df, on=['time', 'thscode'], how='outer')
            
            # 重命名列
            rename_dict = {
                'time': 'dt',
                'thscode': 'trade_code',
                'ths_bond_close_cbond': 'close',
                'ths_new_bond_amt_cbond': 'amount',
                'ths_pure_bond_premium_rate_cbond': 'pure_premium',
                'ths_pure_bond_ytm_cbond': 'ytm',
                'ths_conversion_premium_rate_cbond': 'conv_premium',
                'ths_conver_pe_cbond': 'conv_pe',
                'ths_stock_pe_cbond': 'stock_pe',
                'ths_stock_pb_cbond': 'stock_pb',
                'ths_conver_pb_cbond': 'conv_pb'
            }
            final_df = final_df.rename(columns=rename_dict)
            
            return final_df
            
        except Exception as e:
            self.logger.error(f"采集数据时发生错误: {str(e)}")
            return None
    
    def __del__(self):
        """析构函数"""
        try:
            if self.login_status and THS_AVAILABLE:
                THS_iFinDLogout()
        except:
            pass

# 初始化采集器
collector = THSBondDataCollector()
print(f"数据采集器初始化完成，登录状态: {collector.login_status}")

### 2.3 从数据库读取历史数据

In [ ]:
def get_bond_market_data(start_date: str, end_date: str) -> pd.DataFrame:
    """获取可转债市场数据"""
    with DatabaseConnection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                A.dt,
                A.trade_code,
                A.close,
                A.amount,
                B.ths_bond_balance_bond as balance,
                A.un_conversion_balance
            FROM yq.marketinfo_equity1 A
            INNER JOIN bond.marketinfo_equity B
                ON A.trade_code = B.trade_code AND A.dt = B.dt
            WHERE B.ths_bond_balance_bond > 0
            AND A.trade_code NOT LIKE '%%NQ'
            AND A.dt BETWEEN %s AND %s
        """, (start_date, end_date))
        columns = ['dt', 'trade_code', 'close', 'amount', 'balance', 'un_conversion_balance']
        data = cursor.fetchall()
        df = pd.DataFrame(data, columns=columns)
    return df


def get_bond_characteristics(start_date: str, end_date: str) -> pd.DataFrame:
    """获取可转债特性数据"""
    with DatabaseConnection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT 
                dt,
                trade_code,
                close,
                amount,
                conv_premium as ths_conversion_premium_rate_cbond,
                pure_premium as ths_pure_bond_premium_rate_cbond,
                ytm as ths_pure_bond_ytm_cbond
            FROM yq.marketinfo_equity1
            WHERE dt BETWEEN %s AND %s
            AND trade_code NOT LIKE '%%NQ'
        """, (start_date, end_date))
        columns = ['dt', 'trade_code', 'close', 'amount', 
                  'ths_conversion_premium_rate_cbond', 
                  'ths_pure_bond_premium_rate_cbond', 
                  'ths_pure_bond_ytm_cbond']
        data = cursor.fetchall()
        df = pd.DataFrame(data, columns=columns)
    return df


def get_industry_bonds() -> pd.DataFrame:
    """获取所有可转债的行业分类信息"""
    with DatabaseConnection() as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT DISTINCT
                A.trade_code,
                A.ths_bond_short_name_bond,
                A.ths_object_the_sw_bond,
                A.ths_stock_code_cbond
            FROM bond.basicinfo_equity A
            WHERE A.trade_code NOT LIKE '%%NQ'
            AND A.ths_object_the_sw_bond IS NOT NULL
        """)
        columns = ['trade_code', 'ths_bond_short_name_bond', 'ths_object_the_sw_bond', 'ths_stock_code_cbond']
        data = cursor.fetchall()
        df = pd.DataFrame(data, columns=columns)
    return df

# 测试数据获取
test_start = '2024-01-01'
test_end = '2024-01-31'
market_data = get_bond_market_data(test_start, test_end)
print(f"市场数据: {len(market_data)} 条记录")
print(f"可转债数量: {market_data['trade_code'].nunique()}")

---

## 3. 数据处理

### 3.1 指标计算

In [ ]:
def calculate_turnover_rate(df: pd.DataFrame) -> pd.DataFrame:
    """计算换手率
    换手率 = 成交金额总和 / 未转股余额总和 * 100
    """
    df = df.copy()
    df['turnover_rate'] = (df['amount'] / df['un_conversion_balance']) * 100
    return df


def calculate_parity_premium(df: pd.DataFrame) -> pd.Series:
    """计算平价溢价率"""
    return ((1 + df['ths_pure_bond_premium_rate_cbond']) / 
            (1 + df['ths_conversion_premium_rate_cbond']) - 1) * 100


def calculate_moving_averages(df: pd.DataFrame, window: int = 60) -> pd.DataFrame:
    """计算移动平均"""
    metrics = ['close', 'ths_conversion_premium_rate_cbond', 
               'ths_pure_bond_premium_rate_cbond', 'turnover_rate']
    
    df = df.copy()
    for col in metrics:
        if col in df.columns:
            df[f'{col}_ma{window}'] = df.groupby('trade_code')[col].transform(
                lambda x: x.rolling(window=window, min_periods=1).mean()
            )
    return df


# 测试计算
market_data = calculate_turnover_rate(market_data)
print(f"换手率计算完成，示例:\n{market_data[['trade_code', 'turnover_rate']].head()}")

### 3.2 可转债分类

In [ ]:
def classify_bonds(df: pd.DataFrame) -> pd.DataFrame:
    """
    对债券进行分类
    
    债性分类（基于纯债溢价率）:
    - 偏债型: 纯债溢价率 < -10%
    - 平衡型: -10% <= 纯债溢价率 <= 10%
    - 偏股型: 纯债溢价率 > 10%
    
    溢价分类（基于价格和转股溢价率）:
    - 双低: 价格 <= 110 且 转股溢价率 <= 20%
    - 低价高溢: 价格 <= 110 且 转股溢价率 > 20%
    - 高价低溢: 价格 > 110 且 转股溢价率 <= 20%
    - 双高: 价格 > 110 且 转股溢价率 > 20%
    """
    df = df.copy()
    
    # 债性分类
    df['偏债型'] = df['ths_pure_bond_premium_rate_cbond'] < -10
    df['平衡型'] = ((df['ths_pure_bond_premium_rate_cbond'] >= -10) & 
                       (df['ths_pure_bond_premium_rate_cbond'] <= 10))
    df['偏股型'] = df['ths_pure_bond_premium_rate_cbond'] > 10
    
    # 溢价分类
    df['双低'] = (df['close'] <= 110) & (df['ths_conversion_premium_rate_cbond'] <= 20)
    df['低价高溢'] = (df['close'] <= 110) & (df['ths_conversion_premium_rate_cbond'] > 20)
    df['高价低溢'] = (df['close'] > 110) & (df['ths_conversion_premium_rate_cbond'] <= 20)
    df['双高'] = (df['close'] > 110) & (df['ths_conversion_premium_rate_cbond'] > 20)
    
    return df


def split_industry(industry_str: str) -> List[str]:
    """拆分行业分类"""
    if pd.isna(industry_str):
        return ['Unknown'] * 3
    parts = industry_str.split('--')
    return parts + ['Unknown'] * (3 - len(parts))


# 测试分类
chars_data = get_bond_characteristics(test_start, test_end)
classified = classify_bonds(chars_data)

# 统计分类结果
nature_stats = {
    '偏债型': classified['偏债型'].sum(),
    '平衡型': classified['平衡型'].sum(),
    '偏股型': classified['偏股型'].sum()
}
print(f"债性分类统计: {nature_stats}")

premium_stats = {
    '双低': classified['双低'].sum(),
    '低价高溢': classified['低价高溢'].sum(),
    '高价低溢': classified['高价低溢'].sum(),
    '双高': classified['双高'].sum()
}
print(f"溢价分类统计: {premium_stats}")

---

## 4. 核心逻辑

### 4.1 指数样本配置

In [ ]:
class IndexSampleConfig:
    """指数样本配置管理"""
    def __init__(self):
        # 债性分类指数样本配置
        self.nature_samples = {
            '偏债型': {'target': 50},
            '平衡型': {'target': 50},
            '偏股型': {'target': 50}
        }
        
        # 溢价分类指数样本配置
        self.premium_samples = {
            '双低': {'target': 10},
            '低价高溢': {'target': 10},
            '高价低溢': {'target': 20},
            '双高': {'target': 10},
            '普通型': {'target': 100}
        }
        
        # 行业指数样本配置
        self.industry_target = 10
        
        # 样本补充策略配置
        self.adjustment_steps = [
            {'premium_range': 2},    # 第一步：放宽溢价率±2%
            {'premium_range': 5},    # 第二步：放宽溢价率±5%
            {'turnover_rate': 0.3},  # 第三步：降低换手率要求到0.3%
            {'turnover_rate': 0.1},  # 第四步：进一步降低换手率到0.1%
            {'dynamic_minimum': True} # 最后：完全放开条件
        ]


sample_config = IndexSampleConfig()
print("指数样本配置初始化完成")

### 4.2 指数计算主类

In [ ]:
class BondIndustryIndex:
    """可转债指数计算器"""
    
    def __init__(self):
        self.logger = logging.getLogger('bond_index')
        # 样本管理参数
        self.ma_window = 60  # 60个交易日移动平均
        self.min_continuous_days = 20  # 入池要求连续达标天数
        self.min_turnover_rate = 0.5  # 最低日均换手率(%)
        self.weight_cap = 0.10  # 单只债券权重上限
        
        # 缓存目录
        self.cache_dir = Path('cache')
        self.cache_dir.mkdir(exist_ok=True)
        
        # 样本配置
        self.sample_config = IndexSampleConfig()
    
    def calculate_index(self, market_data: pd.DataFrame, group_data: pd.DataFrame, 
                        group_by: str) -> pd.DataFrame:
        """计算指数（加权平均）"""
        # 转换日期类型
        market_data = market_data.copy()
        group_data = group_data.copy()
        market_data['dt'] = pd.to_datetime(market_data['dt'])
        group_data['dt'] = pd.to_datetime(group_data['dt'])
        
        # 合并数据
        merged_data = pd.merge(market_data, group_data, on=['dt', 'trade_code'], how='inner')
        merged_data = merged_data.dropna(subset=['close', 'balance'])
        
        # 转换数据类型
        merged_data['close'] = merged_data['close'].astype(float)
        merged_data['balance'] = merged_data['balance'].astype(float)
        merged_data['weighted_close'] = merged_data['close'] * merged_data['balance']
        
        # 计算换手率
        if 'amount' in merged_data.columns and 'un_conversion_balance' in merged_data.columns:
            merged_data['turnover_rate'] = (merged_data['amount'] / merged_data['un_conversion_balance']) * 100
        merged_data['turnover_rate'] = merged_data['turnover_rate'].fillna(0)
        
        # 聚合计算
        result = merged_data.groupby(['dt', group_by]).agg({
            'weighted_close': 'sum',
            'balance': 'sum',
            'trade_code': 'count',
            'turnover_rate': 'mean'
        }).reset_index()
        
        # 计算加权平均价格
        result['close'] = result['weighted_close'] / result['balance']
        result = result.drop('weighted_close', axis=1)
        
        # 重命名列
        result = result.rename(columns={
            'balance': 'total_balance',
            'trade_code': 'component_count'
        })
        
        return result
    
    def generate_index_codes(self, names: List[str], prefix: str) -> Dict[str, str]:
        """生成指数代码"""
        return {
            name: f'{prefix}{str(i+1).zfill(3)}'
            for i, name in enumerate(sorted(names))
        }
    
    def get_qualified_samples(self, market_data: pd.DataFrame, chars_data: pd.DataFrame, 
                              industry_bonds: pd.DataFrame) -> pd.DataFrame:
        """获取合格样本"""
        # 1. 合并数据
        df = pd.merge(market_data, chars_data, on=['dt', 'trade_code'], how='inner', suffixes=('', '_chars'))
        df = pd.merge(df, industry_bonds[['trade_code', 'ths_object_the_sw_bond']], 
                     on='trade_code', how='left')
        df['level1'] = df['ths_object_the_sw_bond'].apply(
            lambda x: x.split('--')[0] if pd.notnull(x) else 'Unknown'
        )
        
        # 2. 计算换手率
        df['turnover_rate'] = (df['amount'] / df['un_conversion_balance']) * 100
        
        # 3. 对候选样本进行分类
        df = classify_bonds(df)
        
        return df


# 初始化指数计算器
index_calculator = BondIndustryIndex()
print("指数计算器初始化完成")

---

## 5. 执行与测试

### 5.1 生成历史指数

In [ ]:
def generate_historical_index(start_date: str, end_date: str, sample_limit: int = None):
    """
    生成历史指数数据
    
    Args:
        start_date: 开始日期
        end_date: 结束日期
        sample_limit: 样本数量限制（用于测试）
    """
    logger.info(f"开始生成历史指数数据: {start_date} 到 {end_date}")
    total_start_time = time.time()
    
    # 1. 获取数据
    logger.info("Step 1: 获取基础数据")
    industry_bonds = get_industry_bonds()
    market_data = get_bond_market_data(start_date, end_date)
    chars_data = get_bond_characteristics(start_date, end_date)
    
    logger.info(f"可转债数量: {len(market_data['trade_code'].unique())}")
    logger.info(f"交易天数: {len(market_data['dt'].unique())}")
    
    # 限制样本数量（用于测试）
    if sample_limit:
        unique_codes = market_data['trade_code'].unique()[:sample_limit]
        market_data = market_data[market_data['trade_code'].isin(unique_codes)]
        chars_data = chars_data[chars_data['trade_code'].isin(unique_codes)]
        logger.info(f"限制样本数量为: {sample_limit}")
    
    # 2. 获取合格样本
    logger.info("Step 2: 获取合格样本")
    qualified_samples = index_calculator.get_qualified_samples(market_data, chars_data, industry_bonds)
    logger.info(f"合格样本数量: {len(qualified_samples['trade_code'].unique())}")
    
    # 3. 生成分类数据
    logger.info("Step 3: 生成分类数据")
    
    # 3.1 债性分类数据
    nature_data = []
    for nature_type in ['偏债型', '平衡型', '偏股型']:
        type_data = qualified_samples[qualified_samples[nature_type]].copy()
        type_data['category'] = nature_type
        nature_data.append(type_data[['dt', 'trade_code', 'category']])
    nature_data = pd.concat(nature_data, ignore_index=True) if nature_data else pd.DataFrame()
    
    # 3.2 溢价分类数据
    premium_data = []
    for premium_type in ['双低', '低价高溢', '高价低溢', '双高']:
        type_data = qualified_samples[qualified_samples[premium_type]].copy()
        type_data['category'] = premium_type
        premium_data.append(type_data[['dt', 'trade_code', 'category']])
    premium_data = pd.concat(premium_data, ignore_index=True) if premium_data else pd.DataFrame()
    
    # 3.3 行业分类数据
    industry_data = qualified_samples[['dt', 'trade_code', 'level1']].copy()
    
    # 4. 计算指数
    logger.info("Step 4: 计算指数")
    results = {}
    
    # 4.1 总体指数
    all_bonds_data = qualified_samples[['dt', 'trade_code']].copy()
    all_bonds_data['category'] = '全部'
    results['全部'] = index_calculator.calculate_index(market_data, all_bonds_data, 'category')
    
    # 4.2 债性分类指数
    if not nature_data.empty:
        results['nature'] = index_calculator.calculate_index(market_data, nature_data, 'category')
    
    # 4.3 溢价分类指数
    if not premium_data.empty:
        results['premium'] = index_calculator.calculate_index(market_data, premium_data, 'category')
    
    # 4.4 行业指数
    results['industry'] = index_calculator.calculate_index(market_data, industry_data, 'level1')
    
    total_time = time.time() - total_start_time
    logger.info(f"历史指数数据生成完成，总耗时: {total_time:.2f}秒")
    
    return results


# 测试：生成最近一个月的指数
test_results = generate_historical_index('2024-01-01', '2024-01-31', sample_limit=50)
print(f"\n指数计算结果:")
for key, df in test_results.items():
    print(f"  {key}: {len(df)} 条记录")

### 5.2 结果验证

In [ ]:
# 显示总体指数结果
if '全部' in test_results and not test_results['全部'].empty:
    all_index = test_results['全部']
    print("总体指数数据:")
    display(all_index.head(10))
    
    # 基本统计
    print(f"\n指数统计:")
    print(f"  日期范围: {all_index['dt'].min()} 到 {all_index['dt'].max()}")
    print(f"  样本数量范围: {all_index['component_count'].min()} - {all_index['component_count'].max()}")
    print(f"  平均收盘价: {all_index['close'].mean():.2f}")
else:
    print("未生成总体指数数据")

---

## 6. 工具函数

### 6.1 数据导出

In [ ]:
def export_to_parquet(df: pd.DataFrame, filename: str, output_dir: str = 'output'):
    """导出数据到Parquet文件"""
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    df.to_parquet(filepath)
    logger.info(f"数据已导出到: {filepath}")
    return filepath


def export_to_csv(df: pd.DataFrame, filename: str, output_dir: str = 'output'):
    """导出数据到CSV文件"""
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=False, encoding='utf-8-sig')
    logger.info(f"数据已导出到: {filepath}")
    return filepath


# 示例导出
if '全部' in test_results and not test_results['全部'].empty:
    export_to_parquet(test_results['全部'], 'bond_all_index.parquet')
    export_to_csv(test_results['全部'], 'bond_all_index.csv')

### 6.2 可视化工具

In [ ]:
def plot_index_trend(index_data: pd.DataFrame, title: str = '指数走势'):
    """绘制指数走势图"""
    plt.figure(figsize=(12, 6))
    plt.plot(index_data['dt'], index_data['close'], 'b-', linewidth=1.5)
    plt.title(title, fontsize=14)
    plt.xlabel('日期', fontsize=12)
    plt.ylabel('指数值', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # 保存图片
    os.makedirs('assets', exist_ok=True)
    filepath = os.path.join('assets', f'{title}.png')
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"图表已保存到: {filepath}")
    return filepath


def plot_sample_distribution(index_data: pd.DataFrame, group_col: str, title: str = '样本分布'):
    """绘制样本分布图"""
    sample_counts = index_data.groupby(group_col)['component_count'].mean().sort_values(ascending=False)
    
    plt.figure(figsize=(12, 6))
    sample_counts.plot(kind='bar')
    plt.title(title, fontsize=14)
    plt.xlabel('分类', fontsize=12)
    plt.ylabel('平均样本数', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    os.makedirs('assets', exist_ok=True)
    filepath = os.path.join('assets', f'{title}.png')
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"图表已保存到: {filepath}")
    return filepath


# 示例绑定
if '全部' in test_results and not test_results['全部'].empty:
    plot_index_trend(test_results['全部'], '可转债总体指数走势')
    
if 'industry' in test_results and not test_results['industry'].empty:
    plot_sample_distribution(test_results['industry'], 'level1', '行业样本分布')

print("可视化工具已加载")

---

## 总结

本项目提供了完整的可转债分析工具链，包括：

1. **数据采集**: 从同花顺iFinD采集可转债行情数据
2. **数据处理**: 计算换手率、溢价率等指标
3. **指数构建**: 构建多维度指数体系
   - Nature类指数（偏债型、平衡型、偏股型）
   - Premium类指数（双低、低价高溢、高价低溢、双高）
   - Industry类指数（各行业）
4. **可视化**: 指数走势和样本分布图表

### 使用说明

1. 确保 `config.py` 中的数据库和同花顺配置正确
2. 运行数据采集: `collector.collect_historical_data(days=365)`
3. 生成指数: `generate_historical_index(start_date, end_date)`
4. 结果保存在 `output/` 目录